# 02 · Retrieval và Generation (Người 2, slide 10–16)

Biến câu hỏi thành truy vấn có bộ lọc, tìm bằng dense, BM25 và hybrid, rerank, rồi sinh câu trả lời có trích dẫn.

**Đầu ra cho người sau:** `src/rag.py`: `search_dense`, `search_bm25`, `rrf`, `ask_rag`.

**Cầu nối sang Người 3:** ở P2-11 câu 3, đáp án đúng từ pandas groupby nằm ngay cạnh nhưng RAG không tính ra. Cần cấu trúc quan hệ và khả năng tính trên toàn tập.

**Cell tuỳ chọn:** P2-08 (rerank) bỏ được nếu máy chậm; slide 13 vẫn nói được bằng sơ đồ phễu.

In [ ]:
%run ./00_setup.ipynb

### P2-01 · Dùng lại chỉ mục và mô hình của Người 1, không embed lại
**ẨN** · slide 10 · Người 2

Mục tiêu: kiểm tra chỉ mục Qdrant của Người 1 và nạp mô hình embedding từ cache.

In [ ]:
# P2-01 · ẨN
df = pd.read_parquet(CLEAN_PATH)
assert qdrant_store.count() > 0, "chạy notebook 01 trước để nạp Qdrant"
_ = vector.embed_query("khởi động mô hình")

### P2-02 · LLM trích bộ lọc, pydantic kiểm tra và đổi 5 tỷ thành 5.000 triệu
**CACHE** · slide 10 · Người 2

Mục tiêu: hiểu câu hỏi: LLM trích bộ lọc thành JSON, pydantic kiểm tra và chuẩn hoá đơn vị.

In [ ]:
# P2-02 · CACHE
assert rag.parse_money_mil("5 tỷ") == 5000 and rag.parse_money_mil("1 tỷ 2") == 1200
asks = ["Căn hộ 2PN ở Thủ Đức dưới 5 tỷ",
        "Nhà Cầu Giấy giá từ 5 tỷ đến 8,5 tỷ, diện tích trên 40m2",
        "Đất ở Đà Nẵng tầm 1 tỷ 2 trở xuống"]
rows = []
for q in asks:
    f, out = rag.parse_filters(q)  # gọi Claude lần đầu, sau đó đọc cache
    rows.append({"câu hỏi": q, "LLM trích": {k: v for k, v in out.data.items() if v is not None},
                 "bộ lọc (giá: triệu đồng)": f.model_dump(exclude_none=True)})
pd.DataFrame(rows)

### P2-03 · Cosine đo hướng, không đo độ dài: (6, 8) vẫn được 1,00
**LIVE** · slide 11 · Người 2

Mục tiêu: tính cosine bằng numpy, đối chiếu sklearn: 1,00 / 0,96 / 0,00 cho q = (3, 4).

In [ ]:
# P2-03 · LIVE
from sklearn.metrics.pairwise import cosine_similarity
q = (3, 4)
for d in [(6, 8), (4, 3), (-4, 3)]:
    mine, ref = rag.cosine(q, d), cosine_similarity([q], [d])[0, 0]
    assert abs(mine - ref) < 1e-12
    print(d, f"numpy {mine:.2f} | sklearn {ref:.2f}")
assert [round(rag.cosine(q, d), 2) for d in [(6, 8), (4, 3), (-4, 3)]] == [1.00, 0.96, 0.00]

### P2-04 · Ngưỡng điểm cắt bớt tin kém liên quan khi tăng k
**CACHE** · slide 11 · Người 2

Mục tiêu: dense search top-k kèm ngưỡng; thử k = 3, 5, 8.

In [ ]:
# P2-04 · CACHE
q, threshold = "Căn hộ 2PN ở Thủ Đức giá bao nhiêu?", 0.80
dense_k = cached("p2_04_dense.pkl", lambda: {k: search_dense(q, k=k, threshold=threshold) for k in (3, 5, 8)})
for k, hits in dense_k.items():
    print(f"k = {k}: {len(hits)} tin có cosine ≥ {threshold}")
rag.hits_table(dense_k[8])

### P2-05 · BM25 thắng câu từ khoá, dense thắng câu diễn đạt [ĐIỀN SAU]
**CACHE** · slide 12 · Người 2

Mục tiêu: BM25 (sparse vector trên Qdrant) với bộ tách từ tiếng Việt; so sánh "sổ hồng 2PN" và "gần trường học" với dense.

In [ ]:
# P2-05 · CACHE
pairs = {"sổ hồng 2PN": ["sổ hồng", "2pn"], "gần trường học": ["gần trường học"]}
both = cached("p2_05_bm25_dense.pkl",
              lambda: {q: {"BM25": search_bm25(q, 5), "dense": search_dense(q, 5)} for q in pairs})
print("tách từ:", rag.tokenize_vi("sổ hồng 2PN gần trường học"))
rows = []
for q, terms in pairs.items():
    for method, hits in both[q].items():
        exact = hits["text"].str.lower().map(lambda t: all(x in t for x in terms)).sum()
        rows.append({"câu": q, "cách": method, "top-5 chứa nguyên cụm": f"{exact}/5",
                     "listing_id": hits["listing_id"].tolist()})
pd.DataFrame(rows)

### P2-06 · Hai bảng xếp hạng gộp thành một bằng RRF
**LIVE** · slide 12 · Người 2

Mục tiêu: chạy toy X, Y, Z (assert) rồi chạy câu thật.

In [ ]:
# P2-06 · LIVE
toy = rrf([["X", "Y", "Z"], ["Y", "Z", "X"]])
assert [d for d, _ in toy] == ["Y", "X", "Z"]
print([(d, f"{s:.5f}") for d, s in toy])

q = "Căn hộ 2PN ở Thủ Đức có sổ hồng"
dense, bm25 = search_dense(q, 20), search_bm25(q, 20)
rank = lambda hits, i: hits.listing_id.tolist().index(i) + 1 if i in set(hits.listing_id) else None
pd.DataFrame([{"listing_id": i, "hạng dense": rank(dense, i), "hạng BM25": rank(bm25, i), "RRF": round(s, 5)}
              for i, s in rrf([dense.listing_id.tolist(), bm25.listing_id.tolist()])[:5]])

### P2-07 · Pre-filter giữ đủ k tin đúng điều kiện, post-filter làm rơi gần hết
**LIVE** · slide 13 · Người 2

Mục tiêu: pre-filter và post-filter trên cùng một câu hỏi có điều kiện cứng.

In [ ]:
# P2-07 · LIVE
q = "Căn hộ 2PN ở Thủ Đức dưới 3 tỷ"
f = rag.QueryFilters(district="Thủ Đức", property_type="Căn hộ chung cư", bedrooms=2, max_price_mil=3000)
k = 10
pre = search_dense(q, k=k, filters=f)                    # lọc trong vector DB rồi lấy top-k
post = rag.post_filter(search_dense(q, k=k), f)          # lấy top-k rồi mới lọc
print(f"pre-filter: {len(pre)}/{k} tin, tất cả thoả điều kiện")
print(f"post-filter: {len(post)}/{k} tin còn lại sau khi lọc")
rag.hits_table(pre).head(5)

### P2-08 · Rerank đẩy tin khớp cả hai yêu cầu lên đầu [ĐIỀN SAU]
**CACHE** · slide 13 · Người 2

Mục tiêu: rerank cross-encoder: top-50 xuống top-5 (cell tuỳ chọn, bỏ nếu máy chậm).

In [ ]:
# P2-08 · CACHE
q = "Căn hộ gần trường học, có hồ bơi"
cands = rag.search_hybrid(q, k=50, candidates=50)
top = cached("p2_08_rerank.pkl", lambda: rag.rerank(q, cands, top=5))
before = {lid: i + 1 for i, lid in enumerate(cands.listing_id)}
pd.DataFrame({"listing_id": top.listing_id, "hạng trước (RRF)": top.listing_id.map(before),
              "hạng sau (rerank)": range(1, len(top) + 1), "điểm rerank": top.rerank.round(3),
              "đoạn": top.text.str.slice(0, 70)})

### P2-09 · Câu trả lời bám nguồn, mỗi ý có [Tin#ID]
**LIVE** · slide 14 · Người 2

Mục tiêu: ghép prompt (chỉ thị, ngữ cảnh, câu hỏi) rồi sinh câu trả lời kèm [Tin#ID].

In [ ]:
# P2-09 · LIVE
q = "Căn hộ 2PN ở Thủ Đức giá bao nhiêu?"
found = rag.retrieve(q)
contexts = rag.format_context(found["hits"])
prompt = rag.build_prompt(q, contexts)
print("CHỈ THỊ:", rag.ANSWER_SYSTEM[:160], "…")
print("PROMPT (rút gọn):", prompt[:400].replace("\n", " "), "…\n")
answer = rag.generate(q, contexts)
print("TRẢ LỜI:\n" + answer.text)
print("\nnguồn được trích:", rag.cited_ids(answer.text), "| nguồn đã đưa vào:", found["hits"].listing_id.tolist())

### P2-10 · Đóng gói thành một hàm ask_rag cho Người 4
**ẨN** · slide 14 · Người 2

Mục tiêu: ask_rag(question) trả về câu trả lời và nguồn (định nghĩa trong src/rag.py).

In [ ]:
# P2-10 · ẨN
_r = ask_rag("Căn hộ 2PN ở Thủ Đức giá bao nhiêu?")
assert {"answer", "sources", "contexts", "latency", "input_tokens", "output_tokens"} <= set(_r)

### P2-11 · RAG trả lời đúng tra cứu và ràng buộc nhưng sai câu tổng hợp
**LIVE** · slide 15–16 · Người 2

Mục tiêu: 3 câu: tra cứu (đúng), có ràng buộc (đúng), tổng hợp (sai có chủ đích); in đáp án pandas groupby.

In [ ]:
# P2-11 · LIVE
three = {"tra cứu": "Nhà 6 tầng có thang máy ở Ngọa Long, Bắc Từ Liêm giá bao nhiêu?",
         "ràng buộc": "Căn hộ chung cư 2 phòng ngủ ở Thủ Đức giá dưới 3 tỷ có căn nào?",
         "tổng hợp": "Ở Hồ Chí Minh, quận nào có giá/m² trung bình thấp nhất (chỉ tính quận có ít nhất 20 tin)?"}
for kind, q in three.items():
    r = ask_rag(q)
    print(f"[{kind}] {q}\n→ {r['answer'][:300]}\n  nguồn: {r['sources']}\n")

g = (df[df.province == "Hồ Chí Minh"].dropna(subset=["district", "price_m2_mil"])
     .groupby("district")["price_m2_mil"].agg(["mean", "count"]).query("count >= 20"))
print(f"ĐÁP ÁN pandas groupby: {g['mean'].idxmin()} ({g['mean'].min():.2f} triệu/m², trên {len(g)} quận)")